# CdpStC MA2020 GoC zero-ica: NEURON vs BrainCell Cell.run

这个 notebook 用 GoC 的 `CdpStC_MA20_GoC.mod` 做最小 zero-`ica` 对照，聚焦：
- `Ci/cai`
- `pump/pumpca`
- `CAM*` 状态
- `pump + pumpca` 守恒

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path('/home/swl/braincell-ion_dyn').resolve()
if str(repo_root) in sys.path:
    sys.path.remove(str(repo_root))
sys.path.insert(0, str(repo_root))

os.environ.setdefault('JAX_PLATFORMS', 'cpu')

import brainstate
import brainunit as u
import matplotlib.pyplot as plt
import numpy as np

import braincell
from braincell import Branch, Cell, Morphology
from braincell.filter import BranchSlice, at
from braincell.mech import Ion, MechanismProbe

print('braincell version:', braincell.__version__)
print('braincell import path:', Path(braincell.__file__).resolve())

brainstate.environ.set(precision=64)

In [ ]:
repo_root = Path(braincell.__file__).resolve().parent.parent
mod_dir = repo_root / 'examples' / 'neuron_compare' / 'Cerebellum_mod' / 'GoC'
mod_file = mod_dir / 'ion' / 'CdpStC_MA20_GoC.mod'
print('repo_root:', repo_root)
print('mod_dir:', mod_dir)
print('mod file exists:', mod_file.exists())

mod_text = mod_file.read_text()
print('INITIAL block preview:')
print(mod_text.split('INITIAL {', 1)[1].split('}', 1)[0][:1800])
print('\nKINETIC block preview:')
print(mod_text.split('KINETIC state {', 1)[1].split('}', 1)[0][:2400])

In [ ]:
dt_ms = 0.05
duration_ms = 40.0
steps = int(duration_ms / dt_ms)
times_ms = np.arange(steps + 1) * dt_ms

temperature_celsius = 25.0
v_init_mV = -60.0
diam_um = 20.0
length_um = 20.0

cam_fields = [
    'CAM0',
    'CAM1C',
    'CAM2C',
    'CAM1N2C',
    'CAM1N',
    'CAM2N',
    'CAM2N1C',
    'CAM1C1N',
    'CAM4',
]

tracked_fields = ['Ci', 'pump', 'pumpca'] + cam_fields
print({
    'dt_ms': dt_ms,
    'duration_ms': duration_ms,
    'temperature_celsius': temperature_celsius,
    'v_init_mV': v_init_mV,
    'length_um': length_um,
    'diam_um': diam_um,
    'tracked_fields': tracked_fields,
})

In [ ]:
from neuron import h, load_mechanisms

if not load_mechanisms(str(mod_dir.resolve())):
    raise RuntimeError(f'NEURON mechanisms not found under {mod_dir!s}; compile the GoC mods first.')
h.load_file('stdrun.hoc')

sec = h.Section(name='soma')
sec.L = length_um
sec.diam = diam_um
sec.nseg = 1
seg = sec(0.5)
sec.insert('CdpStC_MA20_GoC')
mech = seg.CdpStC_MA20_GoC

h.celsius = temperature_celsius
h.dt = dt_ms
h.steps_per_ms = 1.0 / h.dt
h.tstop = duration_ms
h.v_init = v_init_mV

t_vec = h.Vector().record(h._ref_t)
cai_vec = h.Vector().record(seg._ref_cai)
ica_vec = h.Vector().record(seg._ref_ica)
pump_vec = h.Vector().record(mech._ref_pump)
pumpca_vec = h.Vector().record(mech._ref_pumpca)
cam_vectors = {name: h.Vector().record(getattr(mech, f'_ref_{name}')) for name in cam_fields}

h.finitialize(h.v_init)
h.frecord_init()
h.continuerun(h.tstop)

neuron_t_ms = np.asarray(t_vec)
neuron_data = {
    'Ci': np.asarray(cai_vec),
    'ica': np.asarray(ica_vec),
    'pump': np.asarray(pump_vec),
    'pumpca': np.asarray(pumpca_vec),
}
for name, vec in cam_vectors.items():
    neuron_data[name] = np.asarray(vec)
neuron_total = neuron_data['pump'] + neuron_data['pumpca']

print('max |ica|:', float(np.max(np.abs(neuron_data['ica']))))
print('NEURON start/end cai:', float(neuron_data['Ci'][0]), float(neuron_data['Ci'][-1]))
print('NEURON start/end pump:', float(neuron_data['pump'][0]), float(neuron_data['pump'][-1]))
print('NEURON start/end pumpca:', float(neuron_data['pumpca'][0]), float(neuron_data['pumpca'][-1]))
print('NEURON max pump conserve drift:', float(np.max(np.abs(neuron_total - neuron_total[0]))))
for name in cam_fields:
    arr = neuron_data[name]
    print(f'NEURON {name} start/end:', float(arr[0]), float(arr[-1]))

In [ ]:
dt = dt_ms * u.ms
duration = duration_ms * u.ms

soma = Branch.from_lengths(lengths=[length_um] * u.um, radii=[diam_um / 2.0, diam_um / 2.0] * u.um, type='soma')
morpho = Morphology.from_root(soma, name='soma')

cell = Cell(morpho, solver='staggered', V_init=v_init_mV * u.mV)
cell.paint(
    BranchSlice(branch_index=0, prox=0.0, dist=1.0),
    Ion(
        'CdpStC_MA2020_GoC',
        name='ca_stc',
        temp=u.celsius2kelvin(temperature_celsius),
    ),
)
cell.place(
    at('soma', 0.5),
    MechanismProbe(mechanism='ca_stc', field='Ci'),
    MechanismProbe(mechanism='ca_stc', field='pump'),
    MechanismProbe(mechanism='ca_stc', field='pumpca'),
    MechanismProbe(mechanism='ca_stc', field='CAM0'),
    MechanismProbe(mechanism='ca_stc', field='CAM1C'),
    MechanismProbe(mechanism='ca_stc', field='CAM2C'),
    MechanismProbe(mechanism='ca_stc', field='CAM1N2C'),
    MechanismProbe(mechanism='ca_stc', field='CAM1N'),
    MechanismProbe(mechanism='ca_stc', field='CAM2N'),
    MechanismProbe(mechanism='ca_stc', field='CAM2N1C'),
    MechanismProbe(mechanism='ca_stc', field='CAM1C1N'),
    MechanismProbe(mechanism='ca_stc', field='CAM4'),
    MechanismProbe(mechanism='ca_stc', field='vrat'),
    MechanismProbe(mechanism='ca_stc', field='parea'),
    MechanismProbe(mechanism='ca_stc', field='dsq'),
    MechanismProbe(mechanism='ca_stc', field='dsqvol'),
)

with brainstate.environ.context(precision=64):
    cell.init_state()
    print('initial probe Ci (mM):', float(np.asarray(cell.sample_probe('soma(0.5)_ca_stc_Ci').to_decimal(u.mM)).reshape(-1)[0]))
    print('initial probe CAM0 (mM):', float(np.asarray(cell.sample_probe('soma(0.5)_ca_stc_CAM0').to_decimal(u.mM)).reshape(-1)[0]))
    print('initial probe vrat:', float(np.asarray(cell.sample_probe('soma(0.5)_ca_stc_vrat')).reshape(-1)[0]))
    run_result = cell.run(dt=dt, duration=duration)

cell_data = {
    'Ci': np.asarray(run_result.traces['soma(0.5)_ca_stc_Ci'].to_decimal(u.mM)),
    'pump': np.asarray(run_result.traces['soma(0.5)_ca_stc_pump'].to_decimal(u.mol / u.cm ** 2)),
    'pumpca': np.asarray(run_result.traces['soma(0.5)_ca_stc_pumpca'].to_decimal(u.mol / u.cm ** 2)),
}
for name in cam_fields:
    cell_data[name] = np.asarray(run_result.traces[f'soma(0.5)_ca_stc_{name}'].to_decimal(u.mM))
cell_geometry = {
    'vrat': np.asarray(run_result.traces['soma(0.5)_ca_stc_vrat']),
    'parea': np.asarray(run_result.traces['soma(0.5)_ca_stc_parea'].to_decimal(u.um)),
    'dsq': np.asarray(run_result.traces['soma(0.5)_ca_stc_dsq'].to_decimal(u.um ** 2)),
    'dsqvol': np.asarray(run_result.traces['soma(0.5)_ca_stc_dsqvol'].to_decimal(u.um ** 2)),
}
cell_total = cell_data['pump'] + cell_data['pumpca']

print('Cell.run start/end Ci:', float(cell_data['Ci'][0]), float(cell_data['Ci'][-1]))
print('Cell.run start/end pump:', float(cell_data['pump'][0]), float(cell_data['pump'][-1]))
print('Cell.run start/end pumpca:', float(cell_data['pumpca'][0]), float(cell_data['pumpca'][-1]))
print('Cell.run max pump conserve drift:', float(np.max(np.abs(cell_total - cell_total[0]))))
for name in cam_fields:
    arr = cell_data[name]
    print(f'Cell.run {name} start/end:', float(arr[0]), float(arr[-1]))
for name, arr in cell_geometry.items():
    print(f'Cell.run {name} first/last:', float(arr[0]), float(arr[-1]))

In [ ]:
compare_t_ms = times_ms[1:]

def summarize_error(y, ref):
    y = np.asarray(y)
    ref = np.asarray(ref)
    n = min(len(y), len(ref))
    diff = y[:n] - ref[:n]
    return {
        'mae': float(np.mean(np.abs(diff))),
        'rmse': float(np.sqrt(np.mean(diff ** 2))),
        'max_abs': float(np.max(np.abs(diff))),
    }

error_summary = {}
for name in tracked_fields:
    neuron_cmp = neuron_data[name][1:]
    cell_cmp = cell_data[name]
    error_summary[name] = summarize_error(cell_cmp, neuron_cmp[:len(cell_cmp)])

fig, axes = plt.subplots(4, 1, figsize=(9, 15), sharex=True)

axes[0].plot(compare_t_ms, neuron_data['Ci'][1:], label='NEURON cai')
axes[0].plot(compare_t_ms[:len(cell_data['Ci'])], cell_data['Ci'], '-.', label='BrainCell Ci (Cell.run)')
axes[0].set_ylabel('Ci / cai (mM)')
axes[0].legend()

axes[1].plot(compare_t_ms, neuron_data['pump'][1:], label='NEURON pump')
axes[1].plot(compare_t_ms, neuron_data['pumpca'][1:], label='NEURON pumpca')
axes[1].plot(compare_t_ms[:len(cell_data['pump'])], cell_data['pump'], '-.', label='BrainCell pump (Cell.run)')
axes[1].plot(compare_t_ms[:len(cell_data['pumpca'])], cell_data['pumpca'], '-.', label='BrainCell pumpca (Cell.run)')
axes[1].set_ylabel('pump states')
axes[1].legend(ncol=2)

for name in ['CAM0', 'CAM1C', 'CAM2C', 'CAM4']:
    axes[2].plot(compare_t_ms, neuron_data[name][1:], label=f'NEURON {name}')
    axes[2].plot(compare_t_ms[:len(cell_data[name])], cell_data[name], '-.', label=f'BrainCell {name} (Cell.run)')
axes[2].set_ylabel('selected CAM* (mM)')
axes[2].legend(ncol=2, fontsize=8)

axes[3].plot(compare_t_ms, neuron_total[1:], label='NEURON pump + pumpca')
axes[3].plot(compare_t_ms[:len(cell_total)], cell_total, '-.', label='BrainCell total (Cell.run)')
axes[3].axhline(neuron_total[0], color='k', linestyle=':', label='initial total')
axes[3].set_xlabel('time (ms)')
axes[3].set_ylabel('pump conserve total')
axes[3].legend()

plt.suptitle('CdpStC MA2020 GoC zero-ica: NEURON vs BrainCell Cell.run')
plt.tight_layout()
plt.show()

print('Per-field Cell.run vs NEURON error summary:')
for name in tracked_fields:
    print(name, error_summary[name])

print('\nPump conserve drift:')
print('  NEURON:', float(np.max(np.abs(neuron_total - neuron_total[0]))))
print('  BrainCell Cell.run:', float(np.max(np.abs(cell_total - cell_total[0]))))

cam_extra = ['CAM1N2C', 'CAM1N', 'CAM2N', 'CAM2N1C', 'CAM1C1N']
print('\nExtra CAM* end-state comparison:')
for name in cam_extra:
    print(
        name,
        {
            'NEURON_end': float(neuron_data[name][-1]),
            'cell_end': float(cell_data[name][-1]),
        },
    )